In [1]:
import tensorflow as tf
print(tf.__version__)


2026-03-22 16:00:07.984413: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2.16.2


In [2]:
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")

df["label"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

df[["review", "sentiment", "label"]].head()

,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


In [3]:
from sklearn.model_selection import train_test_split

X = df["review"].astype(str)
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 40000
Test size: 10000


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train_tfidf, y_train)

baseline_preds = baseline_model.predict(X_test_tfidf)
baseline_acc = accuracy_score(y_test, baseline_preds)

print("Baseline Accuracy:", baseline_acc)
print(classification_report(y_test, baseline_preds))

Baseline Accuracy: 0.895
              precision    recall  f1-score   support

           0       0.90      0.88      0.89      4961
           1       0.89      0.91      0.90      5039

    accuracy                           0.90     10000
   macro avg       0.90      0.89      0.89     10000
weighted avg       0.90      0.90      0.89     10000



In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 5000
max_len = 200

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

print(X_train_pad.shape, X_test_pad.shape)

(40000, 200) (10000, 200)


In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

lstm_model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    LSTM(64),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history_lstm = lstm_model.fit(
    X_train_pad,
    y_train,
    epochs=3,
    batch_size=32,
    validation_split=0.2
)


Epoch 1/3


/Users/ch/nlp-assignment/venv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1000/1000 ━━━━━━━━━━━━━━━━━━━━ 70s 68ms/step - accuracy: 0.8017 - loss: 0.4250 - val_accuracy: 0.8490 - val_loss: 0.3459
Epoch 2/3
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 69s 69ms/step - accuracy: 0.8829 - loss: 0.2883 - val_accuracy: 0.8810 - val_loss: 0.2850
Epoch 3/3
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 77s 77ms/step - accuracy: 0.9095 - loss: 0.2286 - val_accuracy: 0.8795 - val_loss: 0.2941


In [7]:
lstm_preds = (lstm_model.predict(X_test_pad) > 0.5).astype(int).flatten()

lstm_acc = accuracy_score(y_test, lstm_preds)
print("LSTM Accuracy:", lstm_acc)
print(classification_report(y_test, lstm_preds))

313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step
LSTM Accuracy: 0.8885
              precision    recall  f1-score   support

           0       0.88      0.90      0.89      4961
           1       0.90      0.87      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [8]:
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D

cnn_model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    Conv1D(128, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(1, activation='sigmoid')
])

cnn_model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history_cnn = cnn_model.fit(
    X_train_pad,
    y_train,
    epochs=3,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/3


/Users/ch/nlp-assignment/venv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1000/1000 ━━━━━━━━━━━━━━━━━━━━ 43s 42ms/step - accuracy: 0.8282 - loss: 0.3761 - val_accuracy: 0.8863 - val_loss: 0.2715
Epoch 2/3
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 45s 44ms/step - accuracy: 0.9172 - loss: 0.2107 - val_accuracy: 0.8953 - val_loss: 0.2514
Epoch 3/3
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 42s 42ms/step - accuracy: 0.9602 - loss: 0.1200 - val_accuracy: 0.8901 - val_loss: 0.2778


In [9]:
cnn_preds = (cnn_model.predict(X_test_pad) > 0.5).astype(int).flatten()

cnn_acc = accuracy_score(y_test, cnn_preds)
print("CNN Accuracy:", cnn_acc)
print(classification_report(y_test, cnn_preds))

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step
CNN Accuracy: 0.8909
              precision    recall  f1-score   support

           0       0.88      0.90      0.89      4961
           1       0.90      0.88      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [10]:
results = pd.DataFrame({
    "Model": ["Baseline", "LSTM", "CNN"],
    "Accuracy": [baseline_acc, lstm_acc, cnn_acc]
})

results

,Model,Accuracy
0,Baseline,0.8950
1,LSTM,0.8885
2,CNN,0.8909


In [11]:
results.to_csv("results_summary.csv", index=False)